# Week 47, Day 5: Merging k sorted lists

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w047d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import heapq

march = [3, 9, 20]
april = [1, 10]
may = [5, 6, 30]

print(list(heapq.merge(march, april, may)))
print(sorted(march + april + may) == list(heapq.merge(march, april, may)))

In [ ]:
import heapq

def merge_k(lists):
    heap = [(lst[0], i, 0) for i, lst in enumerate(lists) if lst]
    heapq.heapify(heap)
    out = []
    while heap:
        value, i, j = heapq.heappop(heap)
        out.append(value)
        if j + 1 < len(lists[i]):
            heapq.heappush(heap, (lists[i][j + 1], i, j + 1))
    return out

print(merge_k([[3, 9, 20], [1, 10], [], [5, 6, 30]]))

In [ ]:
import heapq

class Merge:
    def __init__(self, *iterables):
        self._its = [iter(x) for x in iterables]
        self._heap = []
        for i, it in enumerate(self._its):
            self._refill(i)

    def _refill(self, i):
        for value in self._its[i]:              # takes at most one item
            heapq.heappush(self._heap, (value, i))
            break

    def __iter__(self):
        return self

    def __next__(self):
        if not self._heap:
            raise StopIteration
        value, i = heapq.heappop(self._heap)
        self._refill(i)
        return value

class Noisy:
    """An iterator that says when each item is read."""
    def __init__(self, name, values):
        self.name, self.values, self.i = name, values, 0
    def __iter__(self):
        return self
    def __next__(self):
        if self.i == len(self.values):
            raise StopIteration
        self.i += 1
        print(f"  ({self.name} gives {self.values[self.i - 1]})")
        return self.values[self.i - 1]

m = Merge(Noisy("a", [1, 4, 7]), Noisy("b", [2, 3, 9]))
print("first:", next(m))
print("next:", next(m))
print("rest:", list(m))

In [ ]:
import heapq

sales_north = [("2026-01-03", 120), ("2026-01-09", 80)]
sales_south = [("2026-01-01", 50), ("2026-01-09", 95), ("2026-01-20", 40)]
for day, amount in heapq.merge(sales_north, sales_south, key=lambda s: s[0]):
    print(day, amount)
print(list(heapq.merge([9, 5, 1], [8, 2], reverse=True)))

In [ ]:
import heapq

class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def merge_linked(heads):
    heap = [(h.value, i, h) for i, h in enumerate(heads) if h is not None]
    heapq.heapify(heap)
    dummy = tail = Node(None)
    while heap:
        value, i, node = heapq.heappop(heap)
        tail.next = tail = node
        if node.next is not None:
            heapq.heappush(heap, (node.next.value, i, node.next))
    tail.next = None
    return dummy.next

def build(values):
    head = None
    for v in reversed(values):
        head = Node(v, head)
    return head

node, out = merge_linked([build([1, 4, 5]), build([1, 3, 4]), None, build([2, 6])]), []
while node:
    out.append(node.value)
    node = node.next
print(out)

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('readings.txt', 'w', encoding='utf-8') as f:
    f.write('42\n7\n19\n3\n88\n21\n5\n63\n11\n')

In [ ]:
import heapq, os

def external_sort(src, dst, chunk_size):
    runs = []
    with open(src) as f:
        while True:
            chunk = [int(line) for _, line in zip(range(chunk_size), f)]
            if not chunk:
                break
            chunk.sort()
            name = f"run_{len(runs)}.txt"
            with open(name, "w") as r:
                r.writelines(f"{v}\n" for v in chunk)
            runs.append(name)
    files = [open(name) for name in runs]
    with open(dst, "w") as out:
        for value in heapq.merge(*[(int(line) for line in f) for f in files]):
            out.write(f"{value}\n")
    for f, name in zip(files, runs):
        f.close()
        os.remove(name)
    return len(runs)

print("runs:", external_sort("readings.txt", "sorted.txt", 4))
print(open("sorted.txt").read().split())

> **Key idea.** Merge k sorted lists by keeping each list's front item in a min-heap as (value, list index, position): pop the smallest, output it, and push the next item from the same list. That's O(n log k) time with only k items in the heap; the list index refills from the right list, breaks ties (so items or nodes are never compared) and makes the merge stable. For iterators, refill from the iterator; an iterator class makes the merge lazy, like `heapq.merge`, which also takes `key` and `reverse`. Linked lists merge by relinking nodes behind a dummy head. External merge sort writes sorted runs that fit in memory, then k-way merges them.

### Exercise: Merging k lists

Write `merge_k(lists)` with a heap of `(value, list index, position)` entries (no `sorted`, `sort` or `heapq.merge`). Then `kth_smallest(lists, k)`: the kth smallest value overall (k from 1), stopping after k pops, with `IndexError` if there are fewer than k values.

In [ ]:
import heapq

def merge_k(lists):
    ___

def kth_smallest(lists, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_merge():\n    \"merging\"\n    assert merge_k([[3, 9, 20], [1, 10], [], [5, 6, 30]]) == [1, 3, 5, 6, 9, 10, 20, 30]\n    assert merge_k([]) == [] and merge_k([[], []]) == []\n    rng = random.Random(1)\n    for _ in range(100):\n        lists = [sorted(rng.randint(0, 50) for _ in range(rng.randint(0, 10))) for _ in range(rng.randint(0, 8))]\n        flat = sorted(v for l in lists for v in l)\n        assert merge_k(lists) == flat\n        if flat:\n            k = rng.randint(1, len(flat))\n            assert kth_smallest(lists, k) == flat[k - 1]\n    try:\n        kth_smallest([[1], [2]], 3)\n        assert False\n    except IndexError:\n        pass\n    assert \"sorted(\" not in source and \".sort(\" not in source and \"merge(\" not in source.replace(\"merge_k(\", \"\")")

### Exercise: A lazy merge

Write the iterator class `Merge(*iterables, key=None)`: it merges any sorted iterables lazily, reading each input only when needed, ordered by `key(item)` (the item itself if `key` is `None`), stable among equal keys (the earlier input wins), and never comparing items themselves (so dicts work). No `yield` and no `heapq.merge`. The checks count how many items you read from each input.

In [ ]:
import heapq

class Merge:
    def __init__(self, *iterables, key=None):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nclass Counted:\n    def __init__(self, values):\n        self.values, self.reads = list(values), 0\n    def __iter__(self):\n        return self\n    def __next__(self):\n        if self.reads == len(self.values):\n            raise StopIteration\n        self.reads += 1\n        return self.values[self.reads - 1]\ndef test_merge():\n    \"merging lazily\"\n    a, b = Counted([1, 4, 7]), Counted([2, 3, 9])\n    m = Merge(a, b)\n    assert iter(m) is m and next(m) == 1 and a.reads + b.reads <= 3\n    assert list(m) == [2, 3, 4, 7, 9]\n    rows = Merge([{\"d\": 1, \"n\": \"x\"}, {\"d\": 3, \"n\": \"y\"}], [{\"d\": 1, \"n\": \"z\"}], key=lambda r: r[\"d\"])\n    assert [r[\"n\"] for r in rows] == [\"x\", \"z\", \"y\"]\n    assert list(Merge()) == [] and list(Merge([], [5])) == [5]\n    rng = random.Random(2)\n    for _ in range(100):\n        ins = [sorted((rng.randint(0, 9), rng.random()) for _ in range(rng.randint(0, 6))) for _ in range(rng.randint(0, 5))]\n        assert list(Merge(*ins, key=lambda p: p[0])) == sorted((p for l in ins for p in l), key=lambda p: p[0])\n    assert \"yield\" not in source and \"heapq.merge\" not in source")

### Exercise: Merging linked lists

Write `merge_linked(heads)`: merge k sorted linked lists (of `Node`s) into one by relinking the existing nodes (create none except a dummy head), returning the new head. Then `merge_pairs(heads)`: the same result by merging lists two at a time in rounds, as in Week 19, with a two-list `merge_two(a, b)` that relinks too.

In [ ]:
import heapq

class Node:
    def __init__(self, value, next=None):
        self.value, self.next = value, next

def merge_linked(heads):
    ___

def merge_two(a, b):
    ___

def merge_pairs(heads):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef build(values):\n    head = None\n    for v in reversed(values):\n        head = Node(v, head)\n    return head\ndef to_list(h):\n    out = []\n    while h:\n        out.append(h.value)\n        h = h.next\n    return out\ndef test_linked():\n    \"linked merges\"\n    rng = random.Random(3)\n    for f in (merge_linked, merge_pairs):\n        assert f([]) is None and f([None, None]) is None\n        for _ in range(60):\n            lists = [sorted(rng.randint(0, 30) for _ in range(rng.randint(0, 8))) for _ in range(rng.randint(1, 7))]\n            heads = [build(l) for l in lists]\n            ids = set()\n            for h in heads:\n                while h:\n                    ids.add(id(h))\n                    h = h.next\n            merged = f(heads)\n            assert to_list(merged) == sorted(v for l in lists for v in l), f.__name__\n            got = set()\n            while merged:\n                got.add(id(merged))\n                merged = merged.next\n            assert got == ids, \"relink the original nodes\"")

### Exercise: An external sort

Write `external_sort(src, dst, chunk_size)`: sort a file of whole numbers (one per line; skip blank lines) by writing sorted runs of at most `chunk_size` numbers to files named `run_0.txt`, `run_1.txt`, ..., merging them with `heapq.merge` into `dst` (one number per line), closing and deleting every run file, and returning the number of runs. Never read the whole of `src` at once (no `read()` or `readlines()` on it). An empty input gives an empty `dst` and 0 runs.

In [ ]:
# Run this cell first: it creates the files used below.
with open('readings.txt', 'w', encoding='utf-8') as f:
    f.write('42\n7\n19\n3\n\n88\n21\n5\n63\n11\n')

In [ ]:
import heapq, os

def external_sort(src, dst, chunk_size):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_small():\n    \"the example file\"\n    reset_files()\n    assert external_sort(\"readings.txt\", \"sorted.txt\", 4) == 3\n    assert read_file(\"sorted.txt\").split() == [\"3\", \"5\", \"7\", \"11\", \"19\", \"21\", \"42\", \"63\", \"88\"]\n    assert not [f for f in files_now() if f.startswith(\"run_\")], \"delete the runs\"\ndef test_random():\n    \"bigger files\"\n    rng = random.Random(4)\n    for n, chunk in [(0, 5), (1, 5), (100, 7), (1000, 100), (500, 1000)]:\n        values = [rng.randint(-10**6, 10**6) for _ in range(n)]\n        write_file(\"big.txt\", \"\".join(f\"{v}\\n\" for v in values))\n        runs = external_sort(\"big.txt\", \"out.txt\", chunk)\n        assert runs == -(-n // chunk) and [int(x) for x in read_file(\"out.txt\").split()] == sorted(values)\n        assert not [f for f in files_now() if f.startswith(\"run_\")]\n    body = source.split(\"def external_sort\")[1]\n    assert \".read()\" not in body and \"readlines\" not in body")

### Exercise: Union and intersection

Write `union_all(lists)`: every value appearing in any of k sorted lists, once each, sorted; and `in_all(lists)`: the values appearing in **every** list (once each), sorted. Both use a single k-way merge with a heap (no sets, no `sorted`): duplicates are adjacent in the merged order, so compare each value with the previous one, and for `in_all` count how many **different lists** each value came from.

In [ ]:
import heapq

def union_all(lists):
    ___

def in_all(lists):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_union():\n    \"union and intersection\"\n    lists = [[1, 2, 2, 5, 9], [2, 3, 5, 5], [0, 2, 5, 7]]\n    assert union_all(lists) == [0, 1, 2, 3, 5, 7, 9] and in_all(lists) == [2, 5]\n    assert union_all([]) == [] and in_all([]) == [] and in_all([[1, 2], []]) == []\n    rng = random.Random(5)\n    for _ in range(100):\n        ls = [sorted(rng.randint(0, 15) for _ in range(rng.randint(0, 10))) for _ in range(rng.randint(1, 5))]\n        assert union_all(ls) == sorted(set().union(*map(set, ls)))\n        assert in_all(ls) == sorted(set.intersection(*map(set, ls)))\n    assert \"set(\" not in source and \"sorted(\" not in source")

### Bug hunt: Comparing iterators

This merge puts `(value, iterator)` in the heap. When two values are equal, Python compares the iterators, which it can't do, so this crashes with a `TypeError`. Put the input's index between the value and the iterator so it prints `[1, 2, 2, 3]`.

In [ ]:
import heapq

def merge(*iterables):
    heap = []
    for it in map(iter, iterables):
        for v in it:
            heapq.heappush(heap, (v, it))
            break
    out = []
    while heap:
        v, it = heapq.heappop(heap)
        out.append(v)
        for nxt in it:
            heapq.heappush(heap, (nxt, it))
            break
    return out

print(merge([1, 2], [2, 3]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"an index to break ties\"\n    expect_output(\"[1, 2, 2, 3]\")")

### Bug hunt: Refilling from the wrong list

After each pop, this refills from the list the **position** says rather than the one the value came from: it uses `j` as the list number. Items are repeated, and once a position reaches 2 there's no list number 2, so it crashes with an `IndexError`. Refill from list `i` so it prints `[1, 2, 3, 4, 5, 6]`.

In [ ]:
import heapq

lists = [[1, 4, 5], [2, 3, 6]]
heap = [(lst[0], i, 0) for i, lst in enumerate(lists)]
heapq.heapify(heap)
out = []
while heap:
    value, i, j = heapq.heappop(heap)
    out.append(value)
    if j + 1 < len(lists[j]):
        heapq.heappush(heap, (lists[j][j + 1], j, j + 1))
print(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the same list\"\n    expect_output(\"[1, 2, 3, 4, 5, 6]\")")

### Bug hunt: An empty list

This starts the heap with every list's first item, but one list is empty, so it crashes with an `IndexError`. Skip empty lists so it prints `[1, 2, 3]`.

In [ ]:
import heapq

lists = [[1, 3], [], [2]]
heap = [(lst[0], i, 0) for i, lst in enumerate(lists)]
heapq.heapify(heap)
out = []
while heap:
    value, i, j = heapq.heappop(heap)
    out.append(value)
    if j + 1 < len(lists[i]):
        heapq.heappush(heap, (lists[i][j + 1], i, j + 1))
print(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"skip empty lists\"\n    expect_output(\"[1, 2, 3]\")")

### Puzzle: The merge loop

Arrange the lines into a k-way merge. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
import heapq

# Merging [[2, 8], [1, 9], [5]] with a heap of (value, list index, position).

p = ___   # the heap's first popped entry, as a tuple
q = ___   # the entry pushed right after that pop
r = ___   # the most entries the heap ever holds during this merge
s = ___   # list(heapq.merge([5, 3], [4, 1], reverse=True))
t = ___   # the time for n items in k lists, like "O(n log k)"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == (1, 1, 0) and q == (9, 1, 1) and r == 3, \"p, q and r\"\n    assert s == [5, 4, 3, 1] and t.replace(\" \", \"\") == \"O(nlogk)\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the smallest range

Given k sorted lists of numbers, find the **smallest range** `[lo, hi]` that contains at least one number from **every** list. Smallest means the least `hi − lo`; on a tie, the one with the smaller `lo`.

Use a frontier: hold one number from each list in a min-heap, starting with every list's first number, and track the largest value currently held. The current range is `[heap top, largest]`. Each step, the only way to make it smaller is to raise the minimum, so pop it and push the next number from **that** list (updating the largest); stop as soon as some list runs out, because the range can no longer include it. Check every range you see. O(n log k) for n numbers in total.

Write `smallest_range(lists)` returning `(lo, hi)`; raise `ValueError` if there are no lists or any list is empty. Then `covering_values(lists)`: one value from each list (in list order) that lies within the smallest range, choosing the first such value in each list.

In [ ]:
import heapq

def smallest_range(lists):
    ___

def covering_values(lists):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef brute(lists):\n    best = None\n    for combo in itertools.product(*lists):\n        lo, hi = min(combo), max(combo)\n        if best is None or (hi - lo, lo) < (best[1] - best[0], best[0]):\n            best = (lo, hi)\n    return best\ndef test_examples():\n    \"examples\"\n    lists = [[4, 10, 15, 24, 26], [0, 9, 12, 20], [5, 18, 22, 30]]\n    assert smallest_range(lists) == (20, 24)\n    assert covering_values(lists) == [24, 20, 22]\n    assert smallest_range([[1, 2, 3], [1, 2, 3]]) == (1, 1) and smallest_range([[5]]) == (5, 5)\n    for bad in ([], [[1], []]):\n        try:\n            smallest_range(bad)\n            assert False\n        except ValueError:\n            pass\ndef test_random():\n    \"against brute force\"\n    rng = random.Random(6)\n    for _ in range(200):\n        lists = [sorted(rng.randint(0, 40) for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 4))]\n        lo, hi = smallest_range(lists)\n        assert (lo, hi) == brute(lists), lists\n        vals = covering_values(lists)\n        assert len(vals) == len(lists) and all(lo <= v <= hi and v in l for v, l in zip(vals, lists))")

## Warm-up

The same answer as concatenating and sorting, but `heapq.merge` never builds the combined list: it hands out one item at a time, looking only at the front of each input. Today you'll see how.

## A heap of fronts

To merge k sorted lists, the next item of the output is always the smallest of the k **front** items (the first not yet used in each list). Finding the smallest of k items by scanning them is O(k) per output item, O(nk) in all. With the fronts in a min-heap, each step is a pop and (usually) a push: O(log k).

1. Push `(lists[i][0], i, 0)` for every non-empty list i: the value, which list it came from, and its position.
2. Pop the smallest `(value, i, j)` and output `value`.
3. If list i has an item at `j + 1`, push `(lists[i][j + 1], i, j + 1)`.
4. Repeat until the heap is empty.

The heap never holds more than k entries, so the total is **O(n log k)**. The list index `i` in each entry does two jobs: it tells you which list to refill from, and it breaks ties between equal values, so the comparison never goes past it (positions never need comparing) and equal values come out in list order, which makes the merge **stable**.

Week 19's pairwise merging is also O(n log k) (log k rounds, each touching every item once). The heap version has two advantages: it works in a single pass, which matters for streams, and it holds only one item per input at a time.

## Merging lazily

Lists can be indexed, but files, generators and network streams can only be read in order. For those, keep an **iterator** for each input and put the iterator's index in the heap entry; when an item is popped, ask that iterator for its next one. Done as an iterator class (Week 45 Day 3 style), the merge itself becomes lazy: nothing is read until someone asks for the next item.

(The `Noisy` iterators just make it visible when each item is read.) The `for ... break` in `_refill` is a tidy way to take one item if there is one; `next(it, sentinel)` works too.

`heapq.merge(*iterables, key=None, reverse=False)` is the library version: lazy, stable, with a `key` like `sorted`'s, and `reverse=True` for inputs sorted largest first.

## Linked lists

Merging k sorted **linked** lists (Week 42) is the same algorithm with nodes in the heap instead of positions. No new nodes are needed: relink the existing ones onto a tail, using a dummy head (Week 42 Day 3) so the first node isn't a special case. Nodes can't be compared, so the list index is essential as a tie-breaker.

`tail.next = tail = node` assigns `tail.next` first and then moves `tail` on, both to `node` (Python assigns left to right in a chained assignment).

## Sorting bigger than memory

Suppose a file has a billion numbers and memory holds a hundred million. **External merge sort** is how databases do it:

1. **Runs.** Read the file a chunk at a time (as much as fits in memory), sort each chunk, and write it out as its own sorted file, a **run**.
2. **Merge.** Open every run and k-way merge them with a heap, reading each run one line at a time and writing the output as you go.

The merge only holds one line per run in memory, plus the heap. Disks are read in long sequential passes, which they're good at. (When there are too many runs to open at once, merge them in batches over several passes.)

`zip(range(chunk_size), f)` reads at most `chunk_size` lines: `zip` stops as soon as `range` runs out, without reading a line too many.

## Other uses of the frontier

The pattern underneath today, a heap holding one **frontier** item per sorted source, refilled from the same source after each pop, solves many problems:

- **Smallest pair sums** (yesterday's challenge): each row of the grid is a sorted source.
- **The kth smallest across k sorted lists**: merge, but stop after k pops.
- **Ugly numbers** and similar sequences: generate numbers in order by multiplying the frontier by fixed factors.
- **Shortest paths** (next phase's Dijkstra's algorithm): the frontier is the set of places reached so far, and the heap always says which to explore next.

## Practice

## Quiz

## Daily challenge

## Before you go

Find three sorted lists in your life (bus timetables for three routes, three people's free slots) and merge them by hand, front items only. Tomorrow is the Forge: an event-driven simulation and a log merger, built on everything heaps can do.